In [32]:
!pip install sentence-transformers faiss-cpu pandas openpyxl

In [33]:
from google.colab import files

uploaded = files.upload(")")

Saving MIS_Interview_Questions (1).xlsx to )/MIS_Interview_Questions (1) (1).xlsx


In [34]:
import pandas as pd

df = pd.read_excel("/content/MIS_Interview_Questions (1).xlsx")

df.head()

,Question,Answer,Subject
0,What is MIS?,Management Information System used for reporti...,MIS Basics
1,What is the difference between data and inform...,Data is raw facts; information is processed data.,MIS Basics
2,What is a KPI?,Key Performance Indicator used to measure perf...,MIS Basics
3,What is a dashboard?,Visual representation of business metrics.,MIS Basics
4,What is a report?,Organized presentation of data.,MIS Basics


In [35]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [36]:
questions = df["Question"].tolist()

embeddings = model.encode(questions)

In [37]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

In [38]:
print(index.ntotal)

89


In [39]:
def search(question):

    query_vector = model.encode([question])

    D, I = index.search(
        np.array(query_vector),
        k=1
    )

    return df.iloc[I[0]]

In [40]:
search("What is a KPI?")

,Question,Answer,Subject
2,What is a KPI?,Key Performance Indicator used to measure perf...,MIS Basics


In [41]:
!pip install transformers accelerate bitsandbytes

In [42]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [43]:
# def ask_mis(question):

#     retrieved = search(question)

#     context = "\n".join(
#         retrieved["Answer"].tolist()
#     )

#     prompt = f"""
#     Context:
#     {context}

#     Question:
#     {question}

#     Answer:
#     """

#     inputs = tokenizer(
#         prompt,
#         return_tensors="pt"
#     )

#     output = llm.generate(
#         **inputs,
#         max_new_tokens=100
#     )

#     return tokenizer.decode(
#         output[0],
#         skip_special_tokens=True
#     )

In [59]:
def ask_mis(question):
    question = question.strip().lower()

    result = df[df["Question"].str.lower().str.strip().str.contains(question, na=False)]

    if not result.empty:
        return result.iloc[0]["Answer"]

    return "Question not found."

In [60]:
ask_mis(
    "What is MIS?"
)

'Management Information System used for reporting and decision-making.'

In [46]:
!pip install streamlit

In [47]:
# %%writefile app.py
# import streamlit as st

# st.title("MIS Interview Assistant")

# question = st.text_input(
#     "Ask a Question"
# )

# if st.button("Submit"):
#     answer = ask_mis(question)
#     st.write(answer)

In [48]:
# %%writefile app2.py
# import streamlit as st

# st.title("MIS Interview Assistant")

# question = st.text_input("Ask a Question")

# if st.button("Submit"):
#     answer = ask_mis(question)
#     st.success(answer)

In [49]:
!pip install streamlit pyngrok -q

In [50]:
from pyngrok import ngrok
import subprocess
import time

subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])

time.sleep(10)

print(ngrok.connect(8501))

ERROR:pyngrok.process.ngrok:t=2026-06-26T15:52:03+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-06-26T15:52:03+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"


PyngrokNgrokError: The ngrok process errored on start: authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [61]:
import gradio as gr

demo = gr.Interface(
    fn=ask_mis,
    inputs=gr.Textbox(label="Ask a Question"),
    outputs=gr.Textbox(label="Answer"),
    title="MIS Interview Assistant"
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e2110fce5a2df9a6ed.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
